<a href="https://colab.research.google.com/github/Prashkov1ch/python-ai-Prashkovich-Anna/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_Lab01_Economists_Student_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 1 (экономисты): Регулярные выражения и «массивы» как списки

**Ограничения:** используйте только стандартную библиотеку Python и модуль `re`.  
Массивы в этой работе — обычные **списки** (`list`). Никаких `numpy`, `pandas` и т.п.

**Данные:** используются Excel-файлы бухгалтерской отчётности компаний (10 файлов).  
В этой лабораторной мы читаем отдельные текстовые поля (ИНН, ОКПО, ОКВЭД, адрес, наименование) и применяем к ним regex.

In [6]:
import re
from openpyxl import load_workbook
import os
from google.colab import drive
drive.mount('/content/drive')
PATH = '/content/drive/MyDrive/цифровая кафедра/'  # Лабораторные_Экономисты замените на имя своей папки
os.chdir(PATH)
FILES = ['Копия 1 Атомэнергопром.xlsx', 'Копия 2 Аэрофлот.xlsx', 'Копия 3 Газпром_петрозаводск.xlsx', 'Копия 4 Лукойл.xlsx', 'Копия 5 Роснефть.xlsx', 'Копия 6 Самолет.xlsx', 'Копия 7 Славмо.xlsx', 'Копия 8 Строительная_компания_Век.xlsx', 'Копия 9 ТГК_1.xlsx', 'Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx']

def load_org_info(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Сведения об организации"]
    # В этих файлах значения для ключевых полей находятся в колонке 13 (M)
    def v(row, col=13):
        return ws.cell(row=row, column=col).value
    return {
        "file": xlsx_name,
        "full_name": v(6),
        "inn": v(10),
        "kpp": v(11),
        "okpo": v(12),
        "okved": v(15),
        "address": v(16),
        "ogrn": v(21),
    }

org_list = [load_org_info(fn) for fn in FILES]

# Список строк "код наименование" из отчёта о фин. результатах (для regex-упражнений)
def load_fin_lines(xlsx_name: str, limit_rows=120):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Отчет о финансовых результатах"]
    lines = []
    for r in range(6, limit_rows+1):
        code = ws.cell(row=r, column=16).value  # "Код строки"
        name = ws.cell(row=r, column=5).value   # "Наименование показателя"
        if code is None or name is None:
            continue
        s = f"{str(code).strip()} {str(name).strip()}"
        if re.fullmatch(r"\d{4}", str(code).strip()):
            lines.append(s)
    return lines

fin_lines = load_fin_lines(FILES[0])  # для упражнений достаточно одного файла

print("Загружено компаний:", len(org_list))
print("Пример записи:", org_list[0])
print("Пример строк отчета о фин.результатах:", fin_lines[:5])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Загружено компаний: 10
Пример записи: {'file': 'Копия 1 Атомэнергопром.xlsx', 'full_name': 'АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"', 'inn': '7706664260', 'kpp': '770601001', 'okpo': '81527864', 'okved': '62.09', 'address': '119017, Москва г, ул Ордынка Б., 24', 'ogrn': '1027700058286'}
Пример строк отчета о фин.результатах: ['2110 Выручка4', '2120 Себестоимость продаж', '2100 Валовая прибыль (убыток)', '2210 Коммерческие расходы', '2220 Управленческие расходы']


### Задание 1
Используя `re`, разберите имена файлов вида `"<номер> <название>.xlsx"`.  
Сформируйте список `companies` из кортежей `(номер:int, название:str)` и отсортируйте по номеру.

In [7]:
import re
FILES = ['Копия 1 Атомэнергопром.xlsx', 'Копия 2 Аэрофлот.xlsx', 'Копия 3 Газпром_петрозаводск.xlsx', 'Копия 4 Лукойл.xlsx', 'Копия 5 Роснефть.xlsx', 'Копия 6 Самолет.xlsx', 'Копия 7 Славмо.xlsx', 'Копия 8 Строительная_компания_Век.xlsx', 'Копия 9 ТГК_1.xlsx', 'Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx']

# 1. Создаем регулярное выражение (шаблон поиска)
# Мы ищем: цифры (номер), затем пробелы, затем текст (название), затем .xlsx
# Скобки () создают "группы захвата", чтобы потом вытащить эти части отдельно
pattern = r"(\d+)\s+(.+)\.xlsx"

companies = []

# 2. Проходимся по каждому имени файла из списка FILES
for filename in FILES:
    # 3. Ищем совпадение с шаблоном в имени файла
    match = re.search(pattern, filename)

    # 4. Если совпадение найдено (защита от ошибок)
    if match:
        # 5. Получаем номер из первой группы и превращаем в число (int)
        number = int(match.group(1))

        # 6. Получаем название из второй группы (оставляем строкой)
        name = match.group(2)

        # 7. Добавляем кортеж (номер, название) в наш список
        companies.append((number, name))

# 8. Сортируем список по номеру (первый элемент кортежа x[0])
companies.sort(key=lambda x: x[0])

# 9. Выводим результат, чтобы проверить
print(companies)

[(1, 'Атомэнергопром'), (2, 'Аэрофлот'), (3, 'Газпром_петрозаводск'), (4, 'Лукойл'), (5, 'Роснефть'), (6, 'Самолет'), (7, 'Славмо'), (8, 'Строительная_компания_Век'), (9, 'ТГК_1'), (10, 'ТНС_ЭНЭРГО_Карелия')]


### Задание 2
Проверьте ИНН компаний регулярным выражением. Для юр. лиц ожидается ровно **10 цифр**.  
Выведите список файлов, где ИНН не прошёл проверку.

In [8]:
import re

# 1. Создаем регулярное выражение для проверки ИНН
# ^ и $ означают начало и конец строки, \d{10} — ровно 10 цифр
inn_pattern = r"^\d{10}$"

# 2. Создаем пустой список для файлов с ошибками
invalid_inn_files = []

# 3. Проходимся по каждой организации из загруженного списка
for org in org_list:
    # 4. Получаем значение ИНН и приводим его к строке
    # (на случай если оно загрузилось как число или None)
    inn_value = str(org['inn'])

    # 5. Проверяем, соответствует ли строка шаблону (ровно 10 цифр)
    if not re.fullmatch(inn_pattern, inn_value):
        # 6. Если не соответствует, добавляем имя файла в список ошибок
        invalid_inn_files.append(org['file'])

# 7. Выводим результат проверки
if len(invalid_inn_files) == 0:
    print("Все ИНН прошли проверку (10 цифр).")
else:
    print("Найдены файлы с некорректным ИНН:")
    for f in invalid_inn_files:
        print(f"  - {f}")

Все ИНН прошли проверку (10 цифр).


### Задание 3
Проверьте ОКПО: ожидается ровно **8 цифр**.  
Выведите количество корректных и некорректных значений.

In [9]:
import re

# 1. Создаем регулярное выражение для проверки ОКПО
# ^ и $ означают начало и конец строки, \d{8} — ровно 8 цифр
okpo_pattern = r"^\d{8}$"

# 2. Создаем счетчики для корректных и некорректных значений
correct_count = 0
incorrect_count = 0

# 3. Создаем список для файлов с ошибками (чтобы при необходимости вывести их)
incorrect_okpo_files = []

# 4. Проходимся по каждой организации из загруженного списка
for org in org_list:
    # 5. Получаем значение ОКПО и приводим его к строке
    # (на случай если оно загрузилось как число или None)
    okpo_value = str(org['okpo'])

    # 6. Проверяем, соответствует ли строка шаблону (ровно 8 цифр)
    if re.fullmatch(okpo_pattern, okpo_value):
        # 7. Если соответствует — увеличиваем счетчик корректных
        correct_count += 1
    else:
        # 8. Если не соответствует — увеличиваем счетчик некорректных
        incorrect_count += 1
        # 9. Добавляем имя файла в список ошибок
        incorrect_okpo_files.append(org['file'])

# 10. Выводим итоговую статистику
print("=== Проверка ОКПО ===")
print(f"Корректных значений: {correct_count}")
print(f"Некорректных значений: {incorrect_count}")
print(f"Всего проверено: {correct_count + incorrect_count}")

# 11. Если есть ошибки, выводим список файлов
if incorrect_count > 0:
    print("\nФайлы с некорректным ОКПО:")
    for f in incorrect_okpo_files:
        print(f"  - {f}")

=== Проверка ОКПО ===
Корректных значений: 10
Некорректных значений: 0
Всего проверено: 10


### Задание 4
Из ОКВЭД (пример: `62.09`) извлеките две группы цифр: `section` и `group`.  
Сформируйте список словарей `okved_parts = [{"file":..., "section":.., "group":..}, ...]`.

In [10]:
import re

# 1. Создаем регулярное выражение для разбора ОКВЭД
# (\d+) — первая группа: одна или более цифр (секция)
# \. — точка (экранирована, так как в regex точка — спецсимвол)
# (\d+) — вторая группа: одна или более цифр (группа)
okved_pattern = r"(\d+)\.(\d+)"

# 2. Создаем пустой список для результатов
okved_parts = []

# 3. Проходимся по каждой организации из загруженного списка
for org in org_list:
    # 4. Получаем значение ОКВЭД и приводим его к строке
    okved_value = str(org['okved'])

    # 5. Ищем совпадение с шаблоном в значении ОКВЭД
    match = re.search(okved_pattern, okved_value)

    # 6. Если совпадение найдено
    if match:
        # 7. Создаем словарь с данными: файл, секция, группа
        okved_data = {
            'file': org['file'],           # Имя файла компании
            'section': match.group(1),     # Первая группа цифр (до точки)
            'group': match.group(2)        # Вторая группа цифр (после точки)
        }

        # 8. Добавляем словарь в список результатов
        okved_parts.append(okved_data)
    else:
        # 9. Если шаблон не найден, добавляем запись с None
        okved_data = {
            'file': org['file'],
            'section': None,
            'group': None
        }
        okved_parts.append(okved_data)

# 10. Выводим результат для проверки
print("=== Разбор ОКВЭД ===")
print(f"Всего обработано компаний: {len(okved_parts)}")
print("\nПервые 5 записей:")
for item in okved_parts[:5]:
    print(f"  {item['file']} -> section: {item['section']}, group: {item['group']}")

=== Разбор ОКВЭД ===
Всего обработано компаний: 10

Первые 5 записей:
  Копия 1 Атомэнергопром.xlsx -> section: 62, group: 09
  Копия 2 Аэрофлот.xlsx -> section: 51, group: 10
  Копия 3 Газпром_петрозаводск.xlsx -> section: 35, group: 22
  Копия 4 Лукойл.xlsx -> section: 70, group: 10
  Копия 5 Роснефть.xlsx -> section: 35, group: 22


### Задание 5
Подсчитайте частоты `section` из ОКВЭД (используйте только списки и dict, без pandas).  
Результат: словарь `freq`, где ключ — section, значение — количество компаний.

In [11]:
# 1. Создаем пустой словарь для хранения частот
# Ключ словаря — номер секции (например, '62'), значение — количество компаний
freq = {}

# 2. Проходимся по каждому элементу списка okved_parts (результат Задания 4)
for item in okved_parts:
    # 3. Получаем значение секции из текущего словаря
    section = item['section']

    # 4. Проверяем, есть ли уже такая секция в словаре freq
    if section in freq:
        # 5. Если есть — увеличиваем счетчик на 1
        freq[section] += 1
    else:
        # 6. Если нет — создаем новую запись со значением 1
        freq[section] = 1

# 7. Выводим итоговую статистику
print("=== Частоты секций ОКВЭД ===")
print(f"Всего уникальных секций: {len(freq)}")
print("\nРаспределение по секциям:")

# 8. Сортируем секции по номеру для красивого вывода
for section in sorted(freq.keys()):
    print(f"  Секция {section}: {freq[section]} комп.")

# 9. Находим самую частую секцию
max_section = max(freq, key=freq.get)
print(f"\nСамая частая секция: {max_section} ({freq[max_section]} компаний)")

=== Частоты секций ОКВЭД ===
Всего уникальных секций: 6

Распределение по секциям:
  Секция 10: 1 комп.
  Секция 35: 3 комп.
  Секция 41: 1 комп.
  Секция 51: 1 комп.
  Секция 62: 2 комп.
  Секция 70: 2 комп.

Самая частая секция: 35 (3 компаний)


### Задание 6
Из адреса извлеките почтовый индекс (6 цифр) и город (слово перед `г,`).  
Выведите 5 примеров в формате: `файл -> индекс, город`.

In [12]:
import re

# 1. Создаем регулярное выражение для разбора адреса
# (\d{6}) — первая группа: ровно 6 цифр (индекс)
# .*? — любые символы (не жадно, до минимального совпадения)
# (\S+) — вторая группа: слово без пробелов (название города)
# \s*г, — пробелы и "г," после названия города
address_pattern = r"(\d{6}).*?(\S+)\s*г,"

# 2. Создаем пустой список для результатов
addresses = []

# 3. Проходимся по каждой организации из загруженного списка
for org in org_list:
    # 4. Получаем значение адреса и приводим его к строке
    address_value = str(org['address'])

    # 5. Ищем совпадение с шаблоном в адресе
    match = re.search(address_pattern, address_value)

    # 6. Если совпадение найдено
    if match:
        # 7. Создаем словарь с данными: файл, индекс, город
        address_data = {
            'file': org['file'],        # Имя файла компании
            'index': match.group(1),    # Первая группа (6 цифр)
            'city': match.group(2)      # Вторая группа (город)
        }

        # 8. Добавляем словарь в список результатов
        addresses.append(address_data)
    else:
        # 9. Если шаблон не найден, добавляем запись с None
        address_data = {
            'file': org['file'],
            'index': None,
            'city': None
        }
        addresses.append(address_data)

# 10. Выводим результат для проверки (первые 5 записей)
print("=== Извлечение адреса ===")
print(f"Всего обработано компаний: {len(addresses)}")
print("\nПервые 5 записей:")
for item in addresses[:5]:
    print(f"  {item['file']} -> {item['index']}, {item['city']}")

=== Извлечение адреса ===
Всего обработано компаний: 10

Первые 5 записей:
  Копия 1 Атомэнергопром.xlsx -> 119017, Москва
  Копия 2 Аэрофлот.xlsx -> 119019, Москва
  Копия 3 Газпром_петрозаводск.xlsx -> 185011, Петрозаводск
  Копия 4 Лукойл.xlsx -> None, None
  Копия 5 Роснефть.xlsx -> 185011, Петрозаводск


### Задание 7
Нормализуйте полное наименование компании:  
1) удалите кавычки `"`; 2) замените множественные пробелы на один; 3) уберите пробелы по краям.  
Сформируйте список `clean_names` длины 10.

In [13]:
import re

# 1. Создаем пустой список для нормализованных названий
clean_names = []

# 2. Проходимся по каждой организации из загруженного списка
for org in org_list:
    # 3. Получаем полное наименование компании и приводим к строке
    name = str(org['full_name'])

    # 4. Шаг 1: Удаляем кавычки (и двойные, и одинарные)
    name = re.sub(r'["\']', '', name)

    # 5. Шаг 2: Заменяем множественные пробелы на один
    name = re.sub(r'\s+', ' ', name)

    # 6. Шаг 3: Убираем пробелы по краям строки
    name = name.strip()

    # 7. Добавляем нормализованное название в список
    clean_names.append(name)

# 8. Выводим результат для проверки
print("=== Нормализация наименований ===")
print(f"Всего компаний: {len(clean_names)}")
print("\nПервые 5 нормализованных названий:")
for i, name in enumerate(clean_names[:5], 1):
    print(f"  {i}. {name}")

=== Нормализация наименований ===
Всего компаний: 10

Первые 5 нормализованных названий:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ
  3. АКЦИОНЕРНОЕ ОБЩЕСТВО ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК
  4. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО НЕФТЯНАЯ КОМПАНИЯ ЛУКОЙЛ
  5. АКЦИОНЕРНОЕ ОБЩЕСТВО ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК


### Задание 8
Разбейте нормализованное название компании на токены с помощью `re.split`,  
где разделители: пробелы, запятые, точки. Покажите первые 10 токенов для первой компании.

In [14]:
import re

# 1. Создаем регулярное выражение для разделения на токены
# [\s,.]+ означает: один или более символов из набора (пробел, запятая, точка)
split_pattern = r"[\s,.]+"

# 2. Создаем пустой список для хранения всех токенов
all_tokens = []

# 3. Проходимся по каждому нормализованному названию из списка clean_names (Задание 7)
for name in clean_names:
    # 4. Разбиваем строку на токены по нашему шаблону
    tokens = re.split(split_pattern, name)

    # 5. Фильтруем пустые строки (могут появиться если есть пробелы в начале/конце)
    tokens = [t for t in tokens if t]

    # 6. Добавляем список токенов в общий список
    all_tokens.append(tokens)

# 7. Выводим результат для проверки
print("=== Токенизация наименований ===")
print(f"Всего компаний обработано: {len(all_tokens)}")

# 8. Показываем первые 10 токенов для первой компании
print("\nПервые 10 токенов первой компании:")
first_company_tokens = all_tokens[0][:10]
for i, token in enumerate(first_company_tokens, 1):
    print(f"  {i}. {token}")

# 9. Показываем общее количество токенов в первом названии
print(f"\nВсего токенов в первом названии: {len(all_tokens[0])}")

=== Токенизация наименований ===
Всего компаний обработано: 10

Первые 10 токенов первой компании:
  1. АКЦИОНЕРНОЕ
  2. ОБЩЕСТВО
  3. АТОМНЫЙ
  4. ЭНЕРГОПРОМЫШЛЕННЫЙ
  5. КОМПЛЕКС

Всего токенов в первом названии: 5


### Задание 9
Используя `re.finditer`, извлеките из списка `fin_lines` все пары (код, наименование).  
Код — 4 цифры в начале строки. Сформируйте словарь `fin_map[code]=name`.

In [15]:
import re

# 1. Создаем регулярное выражение для извлечения кода и наименования
# ^(\d{4}) — начало строки, первая группа: ровно 4 цифры (код)
# \s+ — один или более пробельных символов (разделитель)
# (.+) — вторая группа: один или более любых символов (наименование)
pattern = r"^(\d{4})\s+(.+)"

# 2. Создаем пустой словарь для результата
fin_map = {}

# 3. Проходимся по каждой строке из списка fin_lines
for line in fin_lines:
    # 4. Используем re.finditer для поиска всех совпадений в строке
    # finditer возвращает итератор объектов Match
    matches = re.finditer(pattern, line)

    # 5. Проходим по всем найденным совпадениям
    for match in matches:
        # 6. Извлекаем код (первая группа) и наименование (вторая группа)
        code = match.group(1)
        name = match.group(2)

        # 7. Добавляем пару в словарь: ключ — код, значение — наименование
        fin_map[code] = name

# 8. Выводим результат для проверки
print("=== Извлечение пар (код, наименование) ===")
print(f"Всего найдено строк: {len(fin_map)}")
print("\nПервые 10 записей словаря fin_map:")

# 9. Выводим первые 10 элементов словаря
for i, (code, name) in enumerate(list(fin_map.items())[:10], 1):
    print(f"  {i}. {code} → {name}")

=== Извлечение пар (код, наименование) ===
Всего найдено строк: 26

Первые 10 записей словаря fin_map:
  1. 2110 → Выручка4
  2. 2120 → Себестоимость продаж
  3. 2100 → Валовая прибыль (убыток)
  4. 2210 → Коммерческие расходы
  5. 2220 → Управленческие расходы
  6. 2200 → Прибыль (убыток) от продаж
  7. 2310 → Доходы от участия в других организациях
  8. 2320 → Проценты к получению
  9. 2330 → Проценты к уплате
  10. 2340 → Прочие доходы


### Задание 10
В строках `fin_lines` найдите все упоминания слова «прибыль» в любом регистре (IGNORECASE).  
Выведите список строк, где найдено совпадение.

In [16]:
import re

# 1. Создаем регулярное выражение для поиска слова "прибыль"
# re.IGNORECASE делает поиск нечувствительным к регистру
profit_pattern = r"прибыль"

# 2. Создаем пустой список для строк с совпадениями
profit_lines = []

# 3. Проходимся по каждой строке из списка fin_lines
for line in fin_lines:
    # 4. Ищем совпадение с шаблоном в строке
    # re.IGNORECASE (или re.I) игнорирует регистр букв
    if re.search(profit_pattern, line, re.IGNORECASE):
        # 5. Если найдено — добавляем строку в список результатов
        profit_lines.append(line)

# 6. Выводим итоговую статистику
print("=== Поиск упоминаний слова «прибыль» ===")
print(f"Всего строк в fin_lines: {len(fin_lines)}")
print(f"Найдено строк со словом «прибыль»: {len(profit_lines)}")

# 7. Выводим все найденные строки
print("\nСтроки, содержащие «прибыль»:")
for i, line in enumerate(profit_lines, 1):
    print(f"  {i}. {line}")

=== Поиск упоминаний слова «прибыль» ===
Всего строк в fin_lines: 27
Найдено строк со словом «прибыль»: 13

Строки, содержащие «прибыль»:
  1. 2100 Валовая прибыль (убыток)
  2. 2200 Прибыль (убыток) от продаж
  3. 2300 Прибыль (убыток) до налогообложения
  4. 2410 Налог на прибыль5
  5. 2411 в т.ч.:
текущий налог на прибыль
  6. 2412 отложенный налог на прибыль6
  7. 2400 Чистая прибыль (убыток)
  8. 2510 Результат от переоценки внеоборотных активов, не включаемый в чистую прибыль (убыток) периода
  9. 2520 Результат от прочих операций, не включаемый в чистую прибыль (убыток) периода
  10. 2530 Налог на прибыль от операций, результат которых не включается в чистую прибыль (убыток) периода5
  11. 2900 Базовая прибыль (убыток) на акцию
  12. 2910 Разводненная прибыль (убыток) на акцию
  13. 2410 Текущий налог на прибыль8


### Задание 11
Преобразуйте ИНН в формат `ИНН: 7706 664260` (пример: 4 цифры, пробел, 6 цифр).  
Используйте `re.sub` с группами. Выведите 5 примеров.

In [17]:
import re

# 1. Создаем регулярное выражение для захвата частей ИНН
# (\d{4}) — первая группа: первые 4 цифры
# (\d{6}) — вторая группа: последние 6 цифр
inn_pattern = r"(\d{4})(\d{6})"

# 2. Создаем пустой список для форматированных ИНН
formatted_inn = []

# 3. Проходимся по каждой организации из загруженного списка
for org in org_list:
    # 4. Получаем значение ИНН и приводим его к строке
    inn_value = str(org['inn'])

    # 5. Используем re.sub для замены с группами
    # \1 — ссылка на первую группу (4 цифры)
    # \2 — ссылка на вторую группу (6 цифр)
    # Добавляем префикс "ИНН: " и пробел между группами
    formatted = re.sub(inn_pattern, r"ИНН: \1 \2", inn_value)

    # 6. Добавляем форматированную строку в список
    formatted_inn.append(formatted)

# 7. Выводим результат для проверки (первые 5 примеров)
print("=== Форматирование ИНН ===")
print(f"Всего компаний: {len(formatted_inn)}")
print("\nПервые 5 примеров:")
for i, inn in enumerate(formatted_inn[:5], 1):
    print(f"  {i}. {inn}")

=== Форматирование ИНН ===
Всего компаний: 10

Первые 5 примеров:
  1. ИНН: 7706 664260
  2. ИНН: 7712 040126
  3. ИНН: 1001 009551
  4. ИНН: 7708 004767
  5. ИНН: 1001 009551


### Задание 12
Определите организационно-правовую форму по названию:  
- если есть «ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО» → `PAO`  
- иначе если есть «АКЦИОНЕРНОЕ ОБЩЕСТВО» → `AO`  
- иначе → `OTHER`  
Используйте `re.search` с флагом IGNORECASE. Выведите пары: `файл -> форма`.

In [18]:
import re

# 1. Создаем пустой список для результатов
org_forms = []

# 2. Проходимся по каждой организации из загруженного списка
for org in org_list:
    # 3. Получаем полное наименование компании и приводим к строке
    name = str(org['full_name'])

    # 4. Проверяем на "ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО" (приоритет 1)
    # re.IGNORECASE делает поиск нечувствительным к регистру
    if re.search(r"ПУБЛИЧНОЕ\s+АКЦИОНЕРНОЕ\s+ОБЩЕСТВО", name, re.IGNORECASE):
        form = "PAO"

    # 5. Проверяем на "АКЦИОНЕРНОЕ ОБЩЕСТВО" (приоритет 2)
    elif re.search(r"АКЦИОНЕРНОЕ\s+ОБЩЕСТВО", name, re.IGNORECASE):
        form = "AO"

    # 6. Если ничего не найдено
    else:
        form = "OTHER"

    # 7. Добавляем пару (файл, форма) в список результатов
    org_forms.append((org['file'], form))

# 8. Выводим итоговую статистику
print("=== Определение организационно-правовой формы ===")
print(f"Всего компаний: {len(org_forms)}")

# 9. Подсчитываем количество каждой формы
form_counts = {}
for _, form in org_forms:
    form_counts[form] = form_counts.get(form, 0) + 1

print("\nРаспределение по формам:")
for form, count in form_counts.items():
    print(f"  {form}: {count} компаний")

# 10. Выводим все пары файл -> форма
print("\nПары файл -> форма:")
for file, form in org_forms:
    print(f"  {file} -> {form}")

=== Определение организационно-правовой формы ===
Всего компаний: 10

Распределение по формам:
  AO: 6 компаний
  PAO: 4 компаний

Пары файл -> форма:
  Копия 1 Атомэнергопром.xlsx -> AO
  Копия 2 Аэрофлот.xlsx -> PAO
  Копия 3 Газпром_петрозаводск.xlsx -> AO
  Копия 4 Лукойл.xlsx -> PAO
  Копия 5 Роснефть.xlsx -> AO
  Копия 6 Самолет.xlsx -> PAO
  Копия 7 Славмо.xlsx -> AO
  Копия 8 Строительная_компания_Век.xlsx -> AO
  Копия 9 ТГК_1.xlsx -> PAO
  Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx -> AO
